[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nocapchicken/nocapchicken.github.io/blob/fix/audit-findings/notebooks/fetch_data_colab.ipynb)

# Data Fetch + Sanity Check

Scrapes NC DHHS inspection records, rebuilds the feature matrix, and runs a quick sanity check.
Pushes updated data files back to GitHub when done.

**Setup:** Add a Colab secret named `GITHUB_TOKEN_NOCAPCHICKEN` (key icon in left sidebar) with repo write access.

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/nocapchicken.github.io')
BRANCH = 'fix/audit-findings'

if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH,
                    'https://github.com/nocapchicken/nocapchicken.github.io.git',
                    str(REPO)], check=True, env=env)

os.chdir(REPO)
!pip install -q -r requirements.txt

In [ ]:
import sys, os
from pathlib import Path

REPO = Path('/content/nocapchicken.github.io')
BRANCH = 'fix/audit-findings'
os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import publish_artifacts
print(f'Repo ready at {REPO}')

## 1. Fetch inspections (all 100 counties, 2020-2026)

In [ ]:
!python scripts/make_dataset.py --inspections-only --force

## 2. Build feature matrix

In [ ]:
!python scripts/build_features.py

## 3. Sanity check

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load raw year files
import glob
year_files = sorted(glob.glob('data/raw/inspections_*.csv'))
raw = pd.concat([pd.read_csv(f) for f in year_files], ignore_index=True)

print('=== RAW DATA ===')
print(f'Total rows: {len(raw):,}')
print(f'Year files: {len(year_files)}')
print(f'inspection_date null: {raw.inspection_date.isna().sum():,} / {len(raw):,}')
print(f'\nGrade distribution (raw):')
print(raw.grade.value_counts())

# Load features
feat = pd.read_csv('data/processed/features.csv')
print(f'\n=== FEATURE MATRIX ===')
print(f'Total rows: {len(feat):,}')
print(f'Columns: {len(feat.columns)}')
print(f'\nGrade distribution (features):')
print(feat.grade.value_counts())

print(f'\nRows with Google reviews: {(feat.combined_reviews.fillna("").str.len() > 0).sum():,}')
print(f'inspection_date null in features: {feat.inspection_date.isna().sum():,} / {len(feat):,}')

# Binary label
n_flagged = (feat.grade_encoded > 0).sum()
print(f'\nBinary: {len(feat) - n_flagged:,} safe / {n_flagged:,} flagged')
print(f'Imbalance ratio: {(len(feat) - n_flagged) / max(n_flagged, 1):.0f}:1')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Grade distribution
grade_counts = feat.grade.value_counts().reindex(['A', 'B', 'C'])
colors = ['#16a34a', '#d97706', '#dc2626']
axes[0].bar(grade_counts.index, grade_counts.values, color=colors)
for i, (g, v) in enumerate(grade_counts.items()):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=10)
axes[0].set_title('Grade distribution (features.csv)')
axes[0].set_ylabel('Count')

# 2. Score distribution by grade
for grade, color in zip(['A', 'B', 'C'], colors):
    subset = feat[feat.grade == grade].score
    if len(subset) > 0:
        axes[1].hist(subset, bins=30, alpha=0.7, label=f'{grade} (n={len(subset):,})', color=color)
axes[1].set_xlim(60, 102)
axes[1].set_title('Score by grade')
axes[1].set_xlabel('Inspection score')
axes[1].legend()

# 3. Inspections per year (if dates exist)
if feat.inspection_date.notna().any():
    feat['_year'] = pd.to_datetime(feat.inspection_date, errors='coerce').dt.year
    yearly = feat.groupby(['_year', 'grade']).size().unstack(fill_value=0)
    yearly[['B', 'C']].plot(kind='bar', ax=axes[2], color=['#d97706', '#dc2626'])
    axes[2].set_title('B/C grades per year')
    axes[2].set_xlabel('Year')
    axes[2].tick_params(axis='x', rotation=45)
    feat.drop(columns='_year', inplace=True)
else:
    axes[2].text(0.5, 0.5, 'inspection_date\nstill null!', ha='center', va='center',
                 fontsize=16, color='red', transform=axes[2].transAxes)
    axes[2].set_title('BOM fix check: FAILED')

plt.tight_layout()
plt.show()

## 4. Push updated data to GitHub

In [ ]:
data_files = [str(f) for f in sorted(Path('data/raw').glob('inspections_*.csv'))]

publish_artifacts(
    data_files,
    'data: re-fetch inspections with BOM fix — inspection_date now populated',
    branch=BRANCH,
)